# Lab05: RNN Sequence Models (Kaggle Ready)

This notebook runs RNN/LSTM/GRU/BiLSTM/CNN-LSTM/ConvLSTM on MNIST/EMNIST.

- Dataset is downloaded automatically to `/kaggle/working/data` when internet is enabled.
- If internet is disabled, set `use_fake_data=True` in config.

In [ ]:
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

WORK_DIR = Path('/kaggle/working')
DATA_ROOT = WORK_DIR / 'data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def default_worker_count() -> int:
    return 2

def _normalize() -> transforms.Compose:
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

DATASET_CLASSES: Dict[str, int] = {
    'mnist': 10,
    'emnist-letters': 26,
    'emnist-balanced': 47,
    'emnist-byclass': 62,
}

def _select_dataset(name: str, train: bool, use_fake_data: bool = False, size: Optional[int] = None, num_classes: int = 10):
    name_lower = name.lower()
    if use_fake_data:
        return datasets.FakeData(
            size=size or (2048 if train else 512),
            image_size=(1, 28, 28),
            num_classes=num_classes,
            transform=_normalize(),
        )

    if name_lower == 'mnist':
        return datasets.MNIST(root=DATA_ROOT, train=train, download=True, transform=_normalize())
    if name_lower.startswith('emnist-'):
        split = name_lower.split('-', maxsplit=1)[1]
        return datasets.EMNIST(root=DATA_ROOT, split=split, train=train, download=True, transform=_normalize())

    raise ValueError(f'Unsupported dataset: {name}')

def make_loaders(dataset: str = 'mnist', batch_size: int = 128, limit_train: Optional[int] = None, limit_test: Optional[int] = None, use_fake_data: bool = False, num_workers: Optional[int] = None):
    num_classes = DATASET_CLASSES[dataset.lower()]
    train_set = _select_dataset(dataset, train=True, use_fake_data=use_fake_data, size=limit_train, num_classes=num_classes)
    test_set = _select_dataset(dataset, train=False, use_fake_data=use_fake_data, size=limit_test, num_classes=num_classes)

    if limit_train:
        train_set = Subset(train_set, list(range(min(len(train_set), limit_train))))
    if limit_test:
        test_set = Subset(test_set, list(range(min(len(test_set), limit_test))))

    worker_count = num_workers if num_workers is not None else default_worker_count()
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=worker_count, pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=worker_count, pin_memory=torch.cuda.is_available())
    return train_loader, test_loader

def to_sequence(images: torch.Tensor, scan: str = 'row') -> torch.Tensor:
    seq = images.squeeze(1)
    if scan == 'column':
        seq = seq.transpose(1, 2)
    return seq

class SequenceClassifier(nn.Module):
    def __init__(self, rnn_type='rnn', input_size=28, hidden_size=128, num_layers=1, num_classes=10, dropout=0.2, bidirectional=False, scan='row'):
        super().__init__()
        self.scan = scan
        self.rnn_type = rnn_type.lower()
        rnn_cls = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[self.rnn_type]
        self.rnn = rnn_cls(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), num_classes)

    def forward(self, x):
        seq = to_sequence(x, scan=self.scan)
        outputs, _ = self.rnn(seq)
        outputs = self.dropout(outputs[:, -1, :])
        return self.fc(outputs)

class CNNFeatureExtractor(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(hidden_channels, hidden_channels * 2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.encoder(x)

def infer_sequence_feature_dim(encoder: nn.Module, input_shape=(1,1,28,28)):
    with torch.no_grad():
        dummy = torch.zeros(*input_shape)
        feats = encoder(dummy)
    return feats.size(1) * feats.size(3)

class CNNLSTMClassifier(nn.Module):
    def __init__(self, hidden_size=128, num_layers=1, num_classes=10, dropout=0.2):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        sequence_feature_dim = infer_sequence_feature_dim(self.cnn)
        self.lstm = nn.LSTM(input_size=sequence_feature_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        feats = self.cnn(x)
        seq = feats.permute(0, 2, 1, 3).contiguous().view(feats.size(0), feats.size(2), -1)
        outputs, _ = self.lstm(seq)
        outputs = self.dropout(outputs[:, -1, :])
        return self.fc(outputs)

class ConvLSTMCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.conv = nn.Conv2d(input_channels + hidden_channels, 4 * hidden_channels, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x, h_prev, c_prev):
        gates = self.conv(torch.cat([x, h_prev], dim=1))
        i, f, g, o = gates.chunk(4, dim=1)
        i, f, g, o = torch.sigmoid(i), torch.sigmoid(f), torch.tanh(g), torch.sigmoid(o)
        c_next = f * c_prev + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class ConvLSTMClassifier(nn.Module):
    def __init__(self, hidden_channels=32, num_classes=10, dropout=0.1):
        super().__init__()
        self.cell = ConvLSTMCell(input_channels=1, hidden_channels=hidden_channels)
        self.dropout = nn.Dropout2d(dropout)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(hidden_channels, num_classes))

    def forward(self, x):
        batch_size = x.size(0)
        h = x.new_zeros(batch_size, self.cell.hidden_channels, 1, x.size(-1))
        c = x.new_zeros(batch_size, self.cell.hidden_channels, 1, x.size(-1))
        for t in range(x.size(2)):
            row = x[:, :, t:t+1, :]
            h, c = self.cell(row, h, c)
        return self.classifier(self.dropout(h))

@dataclass
class TrainConfig:
    model_name: str = 'lstm'
    dataset: str = 'mnist'
    epochs: int = 2
    batch_size: int = 128
    learning_rate: float = 1e-3
    hidden_size: int = 128
    num_layers: int = 1
    dropout: float = 0.2
    grad_clip: Optional[float] = None
    scan: str = 'row'
    limit_train: Optional[int] = 10000
    limit_test: Optional[int] = 2000
    optimizer: str = 'adam'
    bidirectional: bool = False
    log_gates: bool = True
    use_fake_data: bool = False
    num_workers: int = 2

def build_model(cfg: TrainConfig, num_classes: int):
    name = cfg.model_name.lower()
    if name == 'rnn':
        return SequenceClassifier('rnn', hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, num_classes=num_classes, dropout=cfg.dropout, bidirectional=cfg.bidirectional, scan=cfg.scan)
    if name == 'lstm':
        return SequenceClassifier('lstm', hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, num_classes=num_classes, dropout=cfg.dropout, bidirectional=cfg.bidirectional, scan=cfg.scan)
    if name == 'gru':
        return SequenceClassifier('gru', hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, num_classes=num_classes, dropout=cfg.dropout, bidirectional=cfg.bidirectional, scan=cfg.scan)
    if name == 'bilstm':
        return SequenceClassifier('lstm', hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, num_classes=num_classes, dropout=cfg.dropout, bidirectional=True, scan=cfg.scan)
    if name == 'cnn-lstm':
        return CNNLSTMClassifier(hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, num_classes=num_classes, dropout=cfg.dropout)
    if name == 'convlstm':
        return ConvLSTMClassifier(hidden_channels=cfg.hidden_size, num_classes=num_classes, dropout=cfg.dropout)
    raise ValueError(f'Unknown model: {cfg.model_name}')

def gradient_norms(model):
    vals = []
    for p in model.parameters():
        if p.grad is not None:
            vals.append(p.grad.data.norm(2).item())
    return float(torch.tensor(vals).mean()) if vals else 0.0

def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss_sum += loss.item() * labels.size(0)
            pred = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return loss_sum / total, correct / total

def train_one_epoch(model, loader, optimizer, criterion, grad_clip=None):
    model.train()
    epoch_loss = 0.0
    grad_logs = []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        if grad_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        grad_logs.append(gradient_norms(model))
        optimizer.step()
        epoch_loss += loss.item() * labels.size(0)
    return epoch_loss / len(loader.dataset), sum(grad_logs) / max(len(grad_logs), 1)

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

def run_experiment(cfg: TrainConfig):
    num_classes = DATASET_CLASSES[cfg.dataset.lower()]
    train_loader, test_loader = make_loaders(dataset=cfg.dataset, batch_size=cfg.batch_size, limit_train=cfg.limit_train, limit_test=cfg.limit_test, use_fake_data=cfg.use_fake_data, num_workers=cfg.num_workers)
    model = build_model(cfg, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    if cfg.optimizer == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=cfg.learning_rate, momentum=0.9)
    elif cfg.optimizer == 'rmsprop':
        optimizer = torch.optim.RMSprop(model.parameters(), lr=cfg.learning_rate)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)

    start = time.time()
    hist = {'train_loss': [], 'train_grad_norm': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(cfg.epochs):
        tr_loss, tr_grad = train_one_epoch(model, train_loader, optimizer, criterion, cfg.grad_clip)
        va_loss, va_acc = evaluate(model, test_loader)
        hist['train_loss'].append(tr_loss)
        hist['train_grad_norm'].append(tr_grad)
        hist['val_loss'].append(va_loss)
        hist['val_acc'].append(va_acc)
        print(f'Epoch {epoch+1}/{cfg.epochs} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_acc={va_acc:.4f}')

    metrics = {
        'final_val_acc': hist['val_acc'][-1],
        'final_val_loss': hist['val_loss'][-1],
        'train_loss': hist['train_loss'][-1],
        'train_grad_norm': hist['train_grad_norm'][-1],
        'params': float(parameter_count(model)),
        'duration_sec': float(time.time() - start),
    }
    return model, metrics


In [ ]:
# Edit config as needed
cfg = TrainConfig(
    model_name='lstm',      # rnn | lstm | gru | bilstm | cnn-lstm | convlstm
    dataset='mnist',        # mnist | emnist-letters | emnist-balanced | emnist-byclass
    epochs=2,
    batch_size=128,
    learning_rate=1e-3,
    hidden_size=128,
    num_layers=1,
    dropout=0.2,
    limit_train=10000,
    limit_test=2000,
    optimizer='adam',
    use_fake_data=False,    # Set True if Kaggle internet is disabled
    num_workers=2,
)

print('Config:', cfg)
model, metrics = run_experiment(cfg)

print('\n===== Experiment Summary =====')
for k, v in metrics.items():
    print(f'{k:20s}: {v}')

save_path = WORK_DIR / f'{cfg.model_name}_{cfg.dataset}.pt'
torch.save(model.state_dict(), save_path)
print(f'\nModel saved to: {save_path}')